# Phase 3 : Business KPI CALCULATION

Objective : the objective of this phase to calculate the 10 core business kpi defined in 
    the project breif - the metrics that would actually go into a board repot .each lpi
    is calculated using the exact formula specified and labelled clearly for use in phase 4
    dashboard and management report

In [15]:
import pandas as pd
orders = pd.read_csv("Data/orders DA.csv")
customers = pd.read_csv("Data/customers DA.csv")
returns = pd.read_csv("Data/returns DA.csv")

orders["order_date"] = pd.to_datetime(
    orders["order_date"], errors="coerce"
)

returns["return_date"] = pd.to_datetime(
    returns["return_date"], errors="coerce"
)

# Standardize status values
orders["clean_status"] = (
    orders["status"].astype(str).str.strip().str.title()
)

print("Orders:", orders.shape)
print("Customers:", customers.shape)
print("Returns:", returns.shape)

print(orders["clean_status"].value_counts())

C:\Users\User\AppData\Local\Temp\ipykernel_23040\2013330242.py:6: UserWarning: Parsing dates in DD/MM/YYYY format when dayfirst=False (the default) was specified. This may lead to inconsistently parsed dates! Specify a format to ensure consistent parsing.
  orders["order_date"] = pd.to_datetime(
C:\Users\User\AppData\Local\Temp\ipykernel_23040\2013330242.py:10: UserWarning: Parsing dates in DD/MM/YYYY format when dayfirst=False (the default) was specified. This may lead to inconsistently parsed dates! Specify a format to ensure consistent parsing.
  returns["return_date"] = pd.to_datetime(


Orders: (50000, 20)
Customers: (10000, 17)
Returns: (10000, 10)
Delivered     32499
Cancelled      5996
Shipped        5071
Returned       3933
Processing     2501
Name: clean_status, dtype: int64


In [21]:
# Standardize order status
orders["clean_status"] = (
    orders["status"].astype(str).str.strip().str.title()
)

# 1. Total orders
total_orders = len(orders)

# 2. Count delivered and cancelled orders
delivered_orders = (
    orders["clean_status"] == "Delivered"
).sum()

cancelled_orders = (
    orders["clean_status"] == "Cancelled"
).sum()

# 3. GMV
gmv = orders["final_amount"].sum()

# 4. Net Revenue
net_revenue = orders.loc[
    orders["clean_status"] == "Delivered",
    "final_amount"
].sum()

# 5. Average Order Value
aov = net_revenue / delivered_orders

# 6. Cancellation Rate
cancellation_rate = (
    cancelled_orders / total_orders
) * 100

# 7. Return Rate (provisional)
return_rate = (
    len(returns) / delivered_orders
) * 100

# Display results
print("Total Orders:", total_orders)
print("Delivered Orders:", delivered_orders)
print("Cancelled Orders:", cancelled_orders)
print("GMV:", round(gmv, 2))
print("Net Revenue:", round(net_revenue, 2))
print("AOV:", round(aov, 2))
print("Cancellation Rate:", round(cancellation_rate, 2), "%")
print("Return Rate (provisional):", round(return_rate, 2), "%")

Total Orders: 50000
Delivered Orders: 32499
Cancelled Orders: 5996
GMV: 3147385544.58
Net Revenue: 2053858611.96
AOV: 63197.59
Cancellation Rate: 11.99 %
Return Rate (provisional): 30.77 %


In [22]:
print("Returns columns:")
print(returns.columns.tolist())

print("\nFirst 5 return records:")
display(returns.head())

print("\nDuplicate return IDs:")
print(returns.iloc[:, 0].duplicated().sum())

Returns columns:
['return_id', 'order_id', 'customer_id', 'return_date', 'reason', 'return_amount', 'refund_status', 'refund_date', 'return_condition', 'remarks']

First 5 return records:


,return_id,order_id,customer_id,return_date,reason,return_amount,refund_status,refund_date,return_condition,remarks
0,RET00001,ORD000015,CUST05470,2024-12-23,Size Issue,24167.33,Pending,30-12-2024,Damaged,Customer initiated return within 7 day window
1,RET00002,ORD000017,CUST07581,2023-01-11,Defective Product,11698.27,Refunded,06-11-2023,Damaged,Customer initiated return within 7 day window
2,RET00003,ORD000038,CUST05691,2025-01-05,Wrong Product Delivered,11147.74,Refunded,06-05-2025,Damaged,Customer initiated return within 7 day window
3,RET00004,ORD000054,CUST03170,2025-06-20,Duplicate Order,9284.55,Refunded,24-06-2025,Acceptable,Customer initiated return within 7 day window
4,RET00005,ORD000072,CUST09318,2025-03-16,Duplicate Order,36912.59,Refunded,23-03-2025,Good,Customer initiated return within 7 day window



Duplicate return IDs:
0


# Step 3 :Calculate CLV by customer segment

In [23]:
print(customers.columns.tolist())

['customer_id', 'first_name', 'last_name', 'email', 'phone', 'city', 'state', 'pincode', 'gender', 'age', 'segment', 'registration_date', 'last_login_date', 'is_verified', 'is_active', 'total_orders', 'total_spent']


In [24]:
clv_by_segment = (
    customers.groupby("segment")["total_spent"]
    .mean()
    .sort_values(ascending=False)
)

display(clv_by_segment.round(2).to_frame(
    "Average Customer Spending"
))

,Average Customer Spending
segment,
Premium,532735.00
Regular,361870.72
New,274866.48
Budget,186178.81
Inactive,32249.08


In [25]:
premium = clv_by_segment.get("Premium", np.nan)
regular = clv_by_segment.get("Regular", np.nan)

if pd.notna(regular) and regular != 0:
    ratio = premium / regular
    print("Premium / Regular spending ratio:", round(ratio, 2))
else:
    print("Cannot calculate the ratio: check the segment labels.")

Premium / Regular spending ratio: 1.47


#summary 
1. Premium customers have the highest average spending.
2. Premium customers spend about 1.47 times as much as Regular customers, below your 3× benchmark.
3. Inactive customers have the lowest average spending, so re-engagement campaigns may be worth investigating.

In [26]:
#kpi 4 :Calculate Month-over-Month GMV growth
# Ensure order_date is datetime
orders["order_date"] = pd.to_datetime(
    orders["order_date"], errors="coerce"
)

# Calculate monthly GMV
monthly_gmv = (
    orders.dropna(subset=["order_date"])
    .set_index("order_date")["final_amount"]
    .resample("MS")
    .sum()
)

# Calculate MoM growth percentage
mom_growth = monthly_gmv.pct_change() * 100

# Create monthly KPI table
monthly_kpis = pd.DataFrame({
    "Monthly_GMV": monthly_gmv,
    "MoM_Growth_%": mom_growth
})

display(monthly_kpis.round(2))

,Monthly_GMV,MoM_Growth_%
order_date,,
2023-01-01,2.915500e+07,NaN
2023-02-01,2.944705e+07,1.00
2023-03-01,2.839437e+07,-3.57
2023-04-01,2.799788e+07,-1.40
2023-05-01,2.437563e+07,-12.94
2023-06-01,5.214389e+07,113.92
2023-07-01,1.038245e+08,99.11
2023-08-01,9.890524e+07,-4.74
2023-09-01,1.128542e+08,14.10


# Highest positive MoM growth: June 2023, at +113.92%.

1.Largest MoM decline: July 2025, at −60.33%.

2. Latest month in the dataset: December 2025, with GMV of approximately 25.60 million and MoM growth of −9.99%.
3.GMV is much lower in the second half of 2025 than in much of 2024. This is worth investigating.

In [27]:
#kpi 5 Top Category Revenue Share
import os

print("Files in your project folder:")
print(os.listdir("."))

print("\nOrders columns:")
print(orders.columns.tolist())

Files in your project folder:
['.ipynb_checkpoints', 'Data', 'data_dictionary __ fin.md', 'IndiaKart_Project_PRD__.docx', 'notebook', 'Phase 1_ Data Cleaning_Zuveriya Tamboli .ipynb', 'Phase 1_ Data Cleaning_Zuveriya Tamboli.ipynb', 'Phase 2_ Exploratory Data Analysis_Zuveriya Tamboli.ipynb', 'Untitled.ipynb', '~$diaKart_Project_PRD__.docx']

Orders columns:
['order_id', 'customer_id', 'order_date', 'order_time', 'status', 'city', 'state', 'pincode', 'total_amount', 'gst_amount', 'shipping_charge', 'discount_amount', 'final_amount', 'payment_method', 'shipping_partner', 'tracking_id', 'delivered_date', 'is_cod', 'channel', 'clean_status']


In [29]:
products = pd.read_csv("Data/products DA.csv")
order_items = pd.read_csv("Data/order_items DA.csv")

print("Products columns:")
print(products.columns.tolist())

print("\nOrder Items columns:")
print(order_items.columns.tolist())

Products columns:
['product_id', 'product_name', 'category', 'subcategory', 'brand', 'sku', 'mrp', 'selling_price', 'cost_price', 'gst_rate', 'hsn_code', 'weight_grams', 'supplier_id', 'rating', 'review_count', 'is_active', 'launch_date']

Order Items columns:
['item_id', 'order_id', 'product_id', 'product_name', 'category', 'quantity', 'unit_price', 'gst_rate', 'gst_amount', 'discount_amount', 'total_price']


In [30]:
# 1. Get only delivered orders
delivered_order_ids = orders.loc[
    orders["clean_status"] == "Delivered",
    "order_id"
]

# 2. Filter order items to delivered orders
delivered_items = order_items[
    order_items["order_id"].isin(delivered_order_ids)
].copy()

# 3. Calculate revenue by category
category_revenue = (
    delivered_items.groupby("category")["total_price"]
    .sum()
    .sort_values(ascending=False)
)

# 4. Calculate category revenue share
total_category_revenue = category_revenue.sum()

category_share = (
    category_revenue / total_category_revenue
) * 100

# 5. Create KPI table
category_kpis = pd.DataFrame({
    "Category_Revenue": category_revenue,
    "Revenue_Share_%": category_share
})

display(category_kpis.round(2))

# 6. Check concentration risk
top_category = category_share.idxmax()
top_share = category_share.max()

print("Top Category:", top_category)
print("Top Category Revenue Share:", round(top_share, 2), "%")

if top_share > 60:
    print("Warning: High category concentration risk.")
else:
    print("Top category share is not above the 60% benchmark.")

,Category_Revenue,Revenue_Share_%
category,,
Electronics,1.076374e+09,57.74
Sports & Fitness,3.192419e+08,17.13
Home & Kitchen,1.576279e+08,8.46
Fashion,9.691033e+07,5.20
Automotive,8.063293e+07,4.33
Office Supplies,6.070733e+07,3.26
Toys & Baby,2.990482e+07,1.60
Beauty & Health,2.168397e+07,1.16
Grocery,1.207455e+07,0.65


Top Category: Electronics
Top Category Revenue Share: 57.74 %
Top category share is not above the 60% benchmark.


In [31]:
#kpi 6 Payment Failure Rate

payments = pd.read_csv("Data/payments DA.csv")
print("Payments shape:", payments.shape)
print("\nPayment columns:")
print(payments.columns.tolist())

print("\nFirst 5 records:")
display(payments.head())

Payments shape: (50000, 13)

Payment columns:
['payment_id', 'order_id', 'customer_id', 'payment_date', 'payment_time', 'payment_method', 'amount', 'status', 'transaction_id', 'bank_name', 'gateway', 'refund_amount', 'refund_date']

First 5 records:


,payment_id,order_id,customer_id,payment_date,payment_time,payment_method,amount,status,transaction_id,bank_name,gateway,refund_amount,refund_date
0,PAY000001,ORD000001,CUST02946,22-05-2025,17:10:00,Cash on Delivery,100984.19,Success,TXN5110698474,SBI,Google Pay,0,NaN
1,PAY000002,ORD000002,CUST04232,16-10-2023,05:32:00,UPI,254478.01,Success,TXN4067460377,IndusInd,Google Pay,0,NaN
2,PAY000003,ORD000003,CUST08078,08-10-2023,12:32:00,Debit Card,32557.06,Success,TXN7056710694,Axis Bank,PhonePe,0,NaN
3,PAY000004,ORD000004,CUST09612,02-10-2023,08:05:00,UPI,24836.94,Success,TXN7460208257,Axis Bank,PhonePe,0,NaN
4,PAY000005,ORD000005,CUST03065,15-06-2024,20:19:00,UPI,156689.29,Success,TXN5835043807,PNB,Razorpay,0,NaN


In [32]:
# Inspect likely payment status columns
for col in payments.columns:
    if "status" in col.lower():
        print(f"\n{col}:")
        print(payments[col].value_counts(dropna=False))


status:
Success    48252
Failed      1748
Name: status, dtype: int64


In [33]:
# Count total and failed payments
total_payments = len(payments)

failed_payments = (
    payments["status"].astype(str).str.strip().str.title()
    == "Failed"
).sum()

successful_payments = (
    payments["status"].astype(str).str.strip().str.title()
    == "Success"
).sum()

# Calculate Payment Failure Rate
payment_failure_rate = (
    failed_payments / total_payments
) * 100 if total_payments else 0

print("Total Payments:", total_payments)
print("Failed Payments:", failed_payments)
print("Successful Payments:", successful_payments)
print("Payment Failure Rate:", round(payment_failure_rate, 2), "%")

if payment_failure_rate < 2:
    print("Benchmark achieved: Payment failure rate is below 2%.")
else:
    print("Benchmark not achieved: Payment failure rate is 2% or higher.")

Total Payments: 50000
Failed Payments: 1748
Successful Payments: 48252
Payment Failure Rate: 3.5 %
Benchmark not achieved: Payment failure rate is 2% or higher.


In [35]:
#kpi 7 : Inventory Fill Rate

inventory = pd.read_csv("Data/inventory DA.csv")

print("Inventory shape:", inventory.shape)
print("\nColumns:")
print(inventory.columns.tolist())

print("\nInventory status counts:")
print(inventory["status"].value_counts(dropna=False))

Inventory shape: (1000, 11)

Columns:
['inventory_id', 'product_id', 'warehouse_location', 'quantity_available', 'quantity_reserved', 'reorder_level', 'reorder_quantity', 'last_restocked_date', 'unit_cost', 'total_inventory_value', 'status']

Inventory status counts:
In Stock        922
Low Stock        76
Out of Stock      2
Name: status, dtype: int64


In [36]:
# Count total unique SKUs
total_skus = inventory["product_id"].nunique()

# Count SKUs with available stock
in_stock_skus = inventory.loc[
    inventory["quantity_available"] > 0,
    "product_id"
].nunique()

# Calculate Inventory Fill Rate
inventory_fill_rate = (
    in_stock_skus / total_skus * 100
    if total_skus else 0
)

print("Total SKUs:", total_skus)
print("SKUs with available stock:", in_stock_skus)
print("Inventory Fill Rate:", round(inventory_fill_rate, 2), "%")

if inventory_fill_rate > 95:
    print("Benchmark achieved: Inventory Fill Rate is above 95%.")
else:
    print("Benchmark not achieved: Inventory Fill Rate is 95% or lower.")

Total SKUs: 1000
SKUs with available stock: 998
Inventory Fill Rate: 99.8 %
Benchmark achieved: Inventory Fill Rate is above 95%.


# summary
1.Inventory availability is strong: 998 of 1,000 unique SKUs have positive available stock.
2 .Cancellations need attention: the rate is 1.99 percentage points above the target.
3. Payment failures need investigation: the rate is 1.50 percentage points above the target.
4. Electronics is the largest category: its 57.74% revenue share is close to the 60% concentration threshold.
5. Revenue trends deserve investigation: the late-2025 GMV decline could reflect fewer orders, lower order values, or a change in order-status mix.